# SISH-Aware Feature Fusion for HER2 Amplification Classification
### Proposed improvement over the base paper's colour–texture pipeline

Base paper: *Pintawong et al., "Conformal Prediction for Uncertainty Quantification and Reliable HER2 Status
Classification in Breast Cancer IHC Images", IEEE Access 2025.*

**Problem.** The base paper's handcrafted features (colour, LBP, Haralick) were designed for **IHC** slides,
where HER2 appears as brown membrane staining. This project's dataset is **SISH** (silver in-situ
hybridization): HER2 genes appear as **black/brown signal dots** and the chromosome-17 reference (**CEP17**) as
**red dots**. Pathologists call amplification from the **HER2 / CEP17 signal ratio**, which the paper's features
never measure. They are also computed on images shrunk to 128 × 128 px, where a signal dot is only 1–2 px.

**Contribution (SISH-aware feature fusion).**
1. **Stain features** – colour deconvolution (H / E / DAB channels) statistics.
2. **SISH signal features** – HER2 and CEP17 dots detected at **native resolution**: cluster-aware signal counts,
   **HER2/CEP17 ratio**, densities per image area and per nuclear area, signal-cluster fraction, nuclear area.
3. **Fusion** of these with the paper's 45 features, evaluated by ablation, and classified by the soft-voting
   ensemble (RBF-SVM + LightGBM + Extra Trees) from the previous benchmark.

**Evaluation protocol (leakage-safe).**
duplicate removal → group-aware held-out test set (locked) → ablation & tuning on the training set only with
repeated CV → equal tuning budget for every model → final test with bootstrap CIs, McNemar and corrected
t-tests → conformal prediction → grouped permutation importance → timing.

## 1. Install & import

In [ ]:
!pip install -q mahotas xgboost lightgbm kagglehub

In [ ]:
import os, sys, time, warnings, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from scipy.stats import loguniform

from PIL import Image
from PIL import Image as PILImage
from IPython.display import display
from joblib import Parallel, delayed

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, RandomizedSearchCV, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler, PowerTransformer, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             cohen_kappa_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
WORK_DIR = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.getcwd())
MODEL_DIR = WORK_DIR
FIG_DIR = os.path.join(WORK_DIR, "sish_figures"); os.makedirs(FIG_DIR, exist_ok=True)

N_ITER = 20          # randomized-search budget per model (identical for every model)
CV_FOLDS = 5
CV_REPEATS = 3       # 3 x 5 = 15 paired folds for significance tests
ALPHAS = [0.05, 0.10]
print("Working dir:", WORK_DIR)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=300, bbox_inches="tight")

## 2. Download the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zakaurrehmanmmu/her2-sish-dataset")
print("Path to dataset files:", path)
DATASET_PATH = path

## 3. Class labels and slide groups (same detection as the benchmark notebook)

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
SKIP_IF_PATH_CONTAINS = ("mask",)
SPLIT_NAMES = {"train", "test", "val", "valid", "validation", "training", "testing"}

LABEL_LEVEL = None           # None = auto, or folder level int
GROUP_LEVEL = None           # None = auto, "filename" = filename prefix, -1 = no grouping, or folder level int
MAX_IMAGES_PER_CLASS = None  # e.g. 600 to run faster

root = Path(DATASET_PATH)
all_files = sorted(
    p for p in root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXT
    and not any(w in str(p.relative_to(root)).lower() for w in SKIP_IF_PATH_CONTAINS)
)
rel = [p.relative_to(root).parts[:-1] for p in all_files]
print(f"Total image files: {len(all_files)}")

tree = Counter("/".join(r[:4]) for r in rel)
print("\nFolder (up to 4 levels) -> #images")
for k, v in sorted(tree.items())[:40]:
    print(f"  {k or '.'} -> {v}")

def detect_label_level():
    for d in range(max(len(r) for r in rel)):
        vals = [r[d] if len(r) > d else None for r in rel]
        if None in vals:
            continue
        uniq = set(vals)
        if 2 <= len(uniq) <= 10 and not {u.lower() for u in uniq} <= SPLIT_NAMES:
            return d
    raise ValueError("Could not detect class folder level - set LABEL_LEVEL manually.")

if LABEL_LEVEL is None:
    LABEL_LEVEL = detect_label_level()
labels = [r[LABEL_LEVEL] for r in rel]
n_cls = len(set(labels))

import re
def file_prefix(p):
    return re.split(r"[_\-\s]", p.stem)[0]

def detect_group():
    # 1) a folder level below the class folder with many distinct values
    min_depth = min(len(r) for r in rel)
    for d in range(LABEL_LEVEL + 1, min_depth):
        g = [f"{r[LABEL_LEVEL]}/{r[d]}" for r in rel]
        if 2 * n_cls <= len(set(g)) <= 0.5 * len(g):
            return d
    # 2) filename prefix such as  slide12_patch_0003.png
    pref = [file_prefix(p) for p in all_files]
    if 2 * n_cls <= len(set(pref)) <= len(pref) / 3:
        return "filename"
    return -1

if GROUP_LEVEL is None:
    GROUP_LEVEL = detect_group()

if GROUP_LEVEL == -1:
    groups_raw = [str(i) for i in range(len(all_files))]          # every image its own group
elif GROUP_LEVEL == "filename":
    groups_raw = [l + "/" + file_prefix(p) for l, p in zip(labels, all_files)]
else:
    groups_raw = [f"{r[LABEL_LEVEL]}/{r[GROUP_LEVEL]}" for r in rel]

df = pd.DataFrame({"path": [str(p) for p in all_files], "label": labels, "group": groups_raw})
if MAX_IMAGES_PER_CLASS:
    df = (df.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), MAX_IMAGES_PER_CLASS), random_state=RANDOM_STATE))
            .reset_index(drop=True))

HAS_GROUPS = GROUP_LEVEL != -1
print(f"\nClass folder level: {LABEL_LEVEL}   |   group level: {GROUP_LEVEL}")
print(df["label"].value_counts().to_string())
if HAS_GROUPS:
    print(f"\n{df['group'].nunique()} slide groups found. Groups per class:")
    print(df.groupby("label")["group"].nunique().to_string())
else:
    print("\n⚠ No slide IDs found -> random patch-level split. Accuracy may be optimistic "
          "because patches of one slide can land in both train and test.")

In [ ]:
# Show a few sample images per class
classes_sorted = sorted(df["label"].unique())
n_show = 4
fig, axes = plt.subplots(len(classes_sorted), n_show, figsize=(3 * n_show, 3 * len(classes_sorted)))
axes = np.atleast_2d(axes)
for i, c in enumerate(classes_sorted):
    sample = df[df["label"] == c].sample(min(n_show, (df["label"] == c).sum()), random_state=RANDOM_STATE)
    for j in range(n_show):
        ax = axes[i, j]; ax.axis("off")
        if j < len(sample):
            ax.imshow(Image.open(sample.iloc[j]["path"]).convert("RGB"))
            ax.set_title(c, fontsize=9)
plt.tight_layout(); plt.show()

## 4. The SISH-aware feature extractor

The next cell writes `sish_features.py` — **the same file the Flask website uses**, so training and deployment
compute identical features. 86 features in 4 groups:

| Group | # | What it measures | Resolution |
|---|---|---|---|
| `paper` | 45 | Colour mean/std, LBP (P24,R3), Haralick — **base paper** | 128 × 128 |
| `stain` | 15 | H / E / DAB deconvolution channels: mean, std, p10, p50, p90 | 128 × 128 |
| `fine_lbp` | 10 | LBP (P8,R1) fine texture | 128 × 128 |
| `signal` | 16 | HER2 & CEP17 dot counts, **HER2/CEP17 ratio**, densities, cluster fraction, nuclear area | native (≤ 4 MP) |

**Signal detection** (colour rules measured on real HER2-SISH images):
* **HER2 signal** = dark (grey < 130) **and** not magenta (Lab a\* < 18) **and** not dark-blue chromatin (B − R < 25)
* **CEP17 signal** = strongly red/magenta (Lab a\* > 28, L < 72)
* **Nucleus** = bluish haematoxylin (B − R > 12, grey < 215)
* Blobs are cleaned by morphological opening and filtered by area. A HER2 blob larger than 3 × the typical
  single-dot area is a **signal cluster** and is counted as *area / single-dot area* signals — the same convention
  pathologists use when they estimate clusters.
* Areas are normalised to a reference scale, so images of different sizes give comparable values.

In [ ]:
%%writefile sish_features.py
"""SISH-aware handcrafted features for HER2 status classification.

Feature vector (86 values), in this order:
  [  0: 45]  Paper features      - Color (6) + LBP P24/R3 (26) + Haralick (13) on a 128x128 image
  [ 45: 60]  Stain features      - H / E / DAB deconvolution channels: mean, std, p10, p50, p90
  [ 60: 70]  Fine LBP            - LBP P8/R1 histogram (10)
  [ 70: 86]  SISH signal features - HER2 (black/brown) and CEP17 (red) signal dots detected at
                                    native resolution, their ratio, densities and cluster statistics

The first 70 values are identical to her2_features_v3.npz from the benchmark notebook.
This file is used unchanged by the training notebook, train_models.py and the Flask app.
"""
import numpy as np
import mahotas
from PIL import Image
from scipy import ndimage as ndi
from skimage.color import rgb2hed, rgb2lab
from skimage.feature import local_binary_pattern

Image.MAX_IMAGE_PIXELS = None

IMAGE_SIZE = (128, 128)          # paper / stain / fine-LBP features (as in the earlier notebooks)
MAX_PIXELS = 4_000_000           # signal detection runs at native resolution, capped for speed

# ---- signal-detection parameters (tuned on HER2-SISH 40x images; colours on a 0-255 scale) ----
BLACK_MAX_GRAY = 130             # HER2 signals are dark ...
BLACK_MAX_ASTAR = 18             # ... but not magenta (that is CEP17)
BLACK_MAX_BLUE_EXCESS = 25       # ... and not dark-blue nuclear chromatin (B - R)
RED_MIN_ASTAR = 28               # CEP17 signals are strongly red/magenta in Lab a*
RED_MAX_L = 72
NUC_MIN_BLUE_EXCESS = 12         # haematoxylin-stained nuclei are bluish (B - R) ...
NUC_MAX_GRAY = 215               # ... and darker than background
MIN_DOT_AREA = 6                 # px at reference scale (removes noise)
MAX_DOT_AREA = 2500              # px at reference scale (bigger = artefact / fold)
CLUSTER_FACTOR = 3.0             # blob > 3 x typical single-dot area = cluster of signals
SINGLE_DOT_AREA = 40.0           # typical single signal area at reference scale (fallback)

PAPER_NAMES = ([f"color_{s}_{c}" for s in ("mean", "std") for c in "RGB"]
               + [f"lbp24_{i}" for i in range(26)]
               + ["har_" + n for n in ["ASM", "contrast", "correlation", "variance", "IDM", "sum_avg",
                                       "sum_var", "sum_entropy", "entropy", "diff_var", "diff_entropy",
                                       "IMC1", "IMC2"]])
STAIN_NAMES = [f"stain_{ch}_{s}" for ch in ("H", "E", "DAB") for s in ("mean", "std", "p10", "p50", "p90")]
FINE_LBP_NAMES = [f"lbp8_{i}" for i in range(10)]
SIGNAL_NAMES = ["her2_signals", "cep17_signals", "her2_blobs", "cep17_blobs",
                "her2_cep17_ratio", "log2_ratio",
                "her2_per_mpx", "cep17_per_mpx",
                "her2_per_nuclear_area", "cep17_per_nuclear_area",
                "her2_area_frac", "cep17_area_frac", "nuclear_area_frac",
                "her2_mean_blob_area", "her2_cluster_frac", "cep17_mean_blob_area"]
FEATURE_NAMES = PAPER_NAMES + STAIN_NAMES + FINE_LBP_NAMES + SIGNAL_NAMES
GROUPS = {"paper": slice(0, 45), "stain": slice(45, 60), "fine_lbp": slice(60, 70), "signal": slice(70, 86)}
N_FEATURES = len(FEATURE_NAMES)    # 86


# ------------------------------------------------------------------ paper features (unchanged)
def extract_color_features(img):
    return np.concatenate([np.mean(img, axis=(0, 1)), np.std(img, axis=(0, 1))])


def extract_lbp_features(gray):
    P, R = 24, 3
    lbp = local_binary_pattern(gray, P, R, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist


def extract_haralick_features(gray):
    return mahotas.features.haralick((gray // 8).astype(np.uint8), return_mean=True)


# ------------------------------------------------------------------ stain + fine texture
def extract_stain_features(rgb):
    hed = rgb2hed(rgb)
    out = []
    for ch in range(3):
        c = hed[..., ch].ravel()
        out += [c.mean(), c.std(), *np.percentile(c, [10, 50, 90])]
    return np.array(out)


def extract_lbp_fine(gray):
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), density=True)
    return hist


# ------------------------------------------------------------------ SISH signal detection
def _load_native(image_source):
    img = Image.open(image_source).convert("RGB")
    w, h = img.size
    scale = 1.0
    if w * h > MAX_PIXELS:
        scale = (MAX_PIXELS / (w * h)) ** 0.5
        img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BILINEAR)
    return np.asarray(img), scale


def _blobs(mask, min_area, max_area):
    mask = ndi.binary_opening(mask, structure=np.ones((2, 2)))
    lab, n = ndi.label(mask)
    if n == 0:
        return lab, np.array([], dtype=float)
    areas = ndi.sum(mask, lab, index=np.arange(1, n + 1)).astype(float)
    keep = (areas >= min_area) & (areas <= max_area)
    remap = np.zeros(n + 1, dtype=np.int32)
    remap[1:][keep] = np.arange(1, keep.sum() + 1)
    return remap[lab], areas[keep]


def detect_signals(rgb, scale=1.0):
    """Return HER2 / CEP17 / nuclei masks and blob areas (areas converted to reference-scale px)."""
    a = rgb.astype(np.float32)
    gray = a @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    lab = rgb2lab(rgb)
    astar, L = lab[..., 1], lab[..., 0]
    blue_excess = a[..., 2] - a[..., 0]

    s2 = scale ** 2
    min_a, max_a = max(1.0, MIN_DOT_AREA * s2), MAX_DOT_AREA * s2

    her2_mask = (gray < BLACK_MAX_GRAY) & (astar < BLACK_MAX_ASTAR) & (blue_excess < BLACK_MAX_BLUE_EXCESS)
    cep_mask = (astar > RED_MIN_ASTAR) & (L < RED_MAX_L)
    nuc_mask = (blue_excess > NUC_MIN_BLUE_EXCESS) & (gray < NUC_MAX_GRAY)
    nuc_mask = ndi.binary_opening(nuc_mask, structure=np.ones((3, 3)))

    her2_lab, her2_areas = _blobs(her2_mask, min_a, max_a)
    cep_lab, cep_areas = _blobs(cep_mask, min_a, max_a)
    return {"her2_lab": her2_lab, "cep_lab": cep_lab, "nuc_mask": nuc_mask,
            "her2_areas": her2_areas / s2, "cep_areas": cep_areas / s2, "scale": scale}


def _estimate_signals(areas):
    """Cluster-aware signal count: a large blob counts as several signals (area / single-dot area)."""
    if len(areas) == 0:
        return 0.0, 0.0
    single = np.median(areas[areas <= np.percentile(areas, 50)]) if len(areas) >= 5 else SINGLE_DOT_AREA
    single = max(single, MIN_DOT_AREA)
    cluster = areas > CLUSTER_FACTOR * single
    count = (~cluster).sum() + np.round(areas[cluster] / single).sum()
    cluster_frac = areas[cluster].sum() / areas.sum()
    return float(count), float(cluster_frac)


def signal_measurements(rgb, scale=1.0):
    d = detect_signals(rgb, scale)
    n_px_ref = rgb.shape[0] * rgb.shape[1] / scale ** 2
    her2_n, her2_cluster = _estimate_signals(d["her2_areas"])
    cep_n = float(len(d["cep_areas"]))          # CEP17 signals are single dots: one per blob
    nuc_area_ref = d["nuc_mask"].sum() / scale ** 2
    ratio = (her2_n + 1.0) / (cep_n + 1.0)
    m = {
        "her2_signals": her2_n, "cep17_signals": cep_n,
        "her2_blobs": float(len(d["her2_areas"])), "cep17_blobs": float(len(d["cep_areas"])),
        "her2_cep17_ratio": ratio, "log2_ratio": float(np.log2(ratio)),
        "her2_per_mpx": her2_n / n_px_ref * 1e6, "cep17_per_mpx": cep_n / n_px_ref * 1e6,
        "her2_per_nuclear_area": her2_n / (nuc_area_ref + 1) * 1e4,
        "cep17_per_nuclear_area": cep_n / (nuc_area_ref + 1) * 1e4,
        "her2_area_frac": d["her2_areas"].sum() / n_px_ref,
        "cep17_area_frac": d["cep_areas"].sum() / n_px_ref,
        "nuclear_area_frac": nuc_area_ref / n_px_ref,
        "her2_mean_blob_area": float(d["her2_areas"].mean()) if len(d["her2_areas"]) else 0.0,
        "her2_cluster_frac": her2_cluster,
        "cep17_mean_blob_area": float(d["cep_areas"].mean()) if len(d["cep_areas"]) else 0.0,
    }
    return m, d


# ------------------------------------------------------------------ public API
def extract_feature_vector(image_source):
    """86-value SISH-aware feature vector for one image (path or file-like)."""
    native, scale = _load_native(image_source)
    small = Image.fromarray(native).resize(IMAGE_SIZE)
    rgb = np.asarray(small)
    gray = np.asarray(small.convert("L"))
    m, _ = signal_measurements(native, scale)
    v = np.concatenate([
        extract_color_features(rgb), extract_lbp_features(gray), extract_haralick_features(gray),
        extract_stain_features(rgb), extract_lbp_fine(gray),
        np.array([m[k] for k in SIGNAL_NAMES], dtype=float),
    ])
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)


def extract_all_features(image_source):
    """Paper's 45 features only (baseline)."""
    return extract_feature_vector(image_source)[:45]


def _overlay(native, d, max_side):
    out = native.copy()
    for lab, colour in ((d["her2_lab"], (0, 200, 0)), (d["cep_lab"], (255, 200, 0))):
        mask = lab > 0
        edge = mask & ~ndi.binary_erosion(mask, iterations=1)
        edge = ndi.binary_dilation(edge, iterations=1)
        out[edge] = colour
    img = Image.fromarray(out)
    img.thumbnail((max_side, max_side))
    return img


def analyse_image(image_source, max_side=900):
    """One pass for the website: (86-feature vector, overlay PIL image, signal measurements)."""
    native, scale = _load_native(image_source)
    small = Image.fromarray(native).resize(IMAGE_SIZE)
    rgb = np.asarray(small)
    gray = np.asarray(small.convert("L"))
    m, d = signal_measurements(native, scale)
    v = np.concatenate([
        extract_color_features(rgb), extract_lbp_features(gray), extract_haralick_features(gray),
        extract_stain_features(rgb), extract_lbp_fine(gray),
        np.array([m[k] for k in SIGNAL_NAMES], dtype=float),
    ])
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0), _overlay(native, d, max_side), m


def signal_overlay(image_source, max_side=900):
    """RGB overlay for display: HER2 signals outlined green, CEP17 signals outlined yellow.
    Returns (PIL.Image, measurements dict)."""
    native, scale = _load_native(image_source)
    m, d = signal_measurements(native, scale)
    return _overlay(native, d, max_side), m

In [ ]:
%%writefile her2_models.py
"""Model wrappers shared by train_models.py and app.py (needed to load the saved models)."""
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline


def mi_score(X, y):
    return mutual_info_classif(X, y, random_state=42)


class ColumnModel(BaseEstimator, ClassifierMixin):
    """Applies a model to a fixed subset of the 86 SISH-aware features (optionally with MI feature selection)."""

    def __init__(self, model=None, feat_cols=None, k_select=None):
        self.model, self.feat_cols, self.k_select = model, feat_cols, k_select

    def _make(self):
        m = clone(self.model)
        if self.k_select:
            m = Pipeline([("select", SelectKBest(mi_score, k=self.k_select)), ("model", m)])
        return m

    def fit(self, X, y):
        self.m_ = self._make().fit(np.asarray(X)[:, self.feat_cols], y)
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X):
        return self.m_.predict_proba(np.asarray(X)[:, self.feat_cols])

    def predict(self, X):
        return self.predict_proba(X).argmax(1)

In [ ]:
sys.path.insert(0, os.getcwd())
import importlib, sish_features as F, her2_models
importlib.reload(F); importlib.reload(her2_models)
from her2_models import ColumnModel      # same wrapper the Flask app uses to load the models
G = F.GROUPS
def cols(groups):
    return np.concatenate([np.arange(G[g].start, G[g].stop) for g in groups])
print(F.N_FEATURES, "features:", {k: v.stop - v.start for k, v in G.items()})

### 4.1 Visual check of signal detection
Green outline = HER2 signal / cluster, yellow outline = CEP17 signal. **Check these before trusting the
features**; if detection looks wrong for your images, adjust the thresholds at the top of `sish_features.py`.

In [ ]:
fig, axes = plt.subplots(len(classes_sorted), 2, figsize=(12, 4.2 * len(classes_sorted)))
axes = np.atleast_2d(axes)
rows = []
for i, c in enumerate(classes_sorted):
    sample = df[df["label"] == c].sample(min(2, (df["label"] == c).sum()), random_state=1)
    for j in range(2):
        ax = axes[i, j]; ax.axis("off")
        if j >= len(sample):
            continue
        ov, m = F.signal_overlay(sample.iloc[j]["path"], max_side=700)
        ax.imshow(ov)
        ax.set_title(f"{c}\nHER2={m['her2_signals']:.0f}  CEP17={m['cep17_signals']:.0f}  "
                     f"ratio={m['her2_cep17_ratio']:.2f}", fontsize=9)
plt.tight_layout(); savefig("signal_detection_examples.png"); plt.show()

## 5. Extract features for all images

In [ ]:
FEATURE_CACHE = os.path.join(WORK_DIR, "her2_sish_features.npz")

def safe_extract(p):
    try:
        return F.extract_feature_vector(p)
    except Exception as e:
        print("Skipping:", p, "|", e)
        return None

if os.path.exists(FEATURE_CACHE):
    c = np.load(FEATURE_CACHE, allow_pickle=True)
    X_all, y_raw, groups, paths = c["X"], c["y"], c["g"], c["paths"]
    print("Loaded cached features:", X_all.shape)
else:
    t0 = time.time()
    feats = Parallel(n_jobs=-1, verbose=5)(delayed(safe_extract)(p) for p in df["path"])
    keep = [i for i, f in enumerate(feats) if f is not None]
    X_all = np.array([feats[i] for i in keep], dtype=np.float64)
    y_raw, groups, paths = df["label"].values[keep], df["group"].values[keep], df["path"].values[keep]
    print(f"Extracted {X_all.shape} in {time.time()-t0:.0f}s")

# ---- remove exact duplicate images (identical feature vectors) to prevent train/test leakage ----
_, first_idx = np.unique(np.round(X_all, 8), axis=0, return_index=True)
first_idx = np.sort(first_idx)
n_dup = len(X_all) - len(first_idx)
X_all, y_raw, groups, paths = X_all[first_idx], y_raw[first_idx], groups[first_idx], paths[first_idx]
print(f"Removed {n_dup} duplicate images -> {len(X_all)} unique images")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
CLASS_NAMES = list(label_encoder.classes_)
n_classes = len(CLASS_NAMES)

def find_class(*words):
    for i, c in enumerate(CLASS_NAMES):
        if all(w in c.lower() for w in words):
            return i
    return None
NON_AMP = find_class("non")
AMP = next((i for i, c in enumerate(CLASS_NAMES) if "amp" in c.lower() and i != NON_AMP), None)
NORMAL = find_class("normal")
print("Classes:", dict(enumerate(CLASS_NAMES)), "| AMP:", AMP, "NON_AMP:", NON_AMP, "NORMAL:", NORMAL)
print(pd.Series(y_raw).value_counts().to_string())

## 6. Do the SISH signal features separate the classes?
Kruskal–Wallis test per feature (non-parametric, 3 groups). Small p = the feature differs between classes.

In [ ]:
sig_df = pd.DataFrame(X_all[:, G["signal"]], columns=F.SIGNAL_NAMES)
sig_df["class"] = y_raw
kw = []
for f in F.SIGNAL_NAMES:
    samples = [sig_df.loc[sig_df["class"] == c, f].values for c in CLASS_NAMES]
    h, p = stats.kruskal(*samples)
    kw.append({"feature": f, "H": h, "p-value": p,
               **{f"median[{c}]": np.median(s) for c, s in zip(CLASS_NAMES, samples)}})
kw = pd.DataFrame(kw).set_index("feature").sort_values("H", ascending=False)
display(kw.style.format({"H": "{:.1f}", "p-value": "{:.2e}"}, precision=3))
kw.to_csv(os.path.join(WORK_DIR, "sish_signal_feature_tests.csv"))

show = ["log2_ratio", "her2_per_nuclear_area", "cep17_per_nuclear_area", "her2_cluster_frac"]
fig, axes = plt.subplots(1, len(show), figsize=(4.2 * len(show), 4))
for ax, f in zip(axes, show):
    ax.boxplot([sig_df.loc[sig_df["class"] == c, f] for c in CLASS_NAMES], showfliers=False)
    ax.set_xticks(range(1, n_classes + 1)); ax.set_xticklabels([c[:14] for c in CLASS_NAMES], rotation=20)
    ax.set_title(f"{f}\nKW p = {kw.loc[f, 'p-value']:.1e}", fontsize=10)
if "log2_ratio" in show:
    axes[0].axhline(1.0, ls="--", c="r", lw=1); axes[0].text(0.6, 1.05, "ratio = 2", color="r", fontsize=8)
plt.tight_layout(); savefig("signal_features_by_class.png"); plt.show()

## 7. Locked held-out test set (group-aware, ~20 %)
Everything until section 10 uses the training set only.

In [ ]:
HAS_GROUPS = len(set(groups)) < len(groups)
def make_cv(seed, n_splits=CV_FOLDS):
    if HAS_GROUPS:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

tr_idx, te_idx = next(make_cv(RANDOM_STATE).split(X_all, y, groups))
X_tr, X_te, y_tr, y_te, g_tr = X_all[tr_idx], X_all[te_idx], y[tr_idx], y[te_idx], groups[tr_idx]
if HAS_GROUPS:
    assert not (set(g_tr) & set(groups[te_idx])), "slide leakage"
print(f"Group-aware split: {HAS_GROUPS} | train {len(tr_idx)} | test {len(te_idx)}")
print("Test classes:", dict(Counter(label_encoder.inverse_transform(y_te))))

def predict_labels(m, X):
    return np.asarray(m.predict(X)).ravel().astype(int)

def amp_nonamp_f1(y_true, y_pred):
    if AMP is None or NON_AMP is None:
        return np.nan
    mask = np.isin(y_true, [AMP, NON_AMP])
    return f1_score(y_true[mask], y_pred[mask], labels=[AMP, NON_AMP], average="macro", zero_division=0)

SPLITS = [(a, b) for r in range(CV_REPEATS) for a, b in make_cv(100 + r).split(X_tr, y_tr, g_tr)]

def repeated_cv(make_model, feat_cols):
    def run(a, b):
        m = make_model().fit(X_tr[a][:, feat_cols], y_tr[a])
        p = predict_labels(m, X_tr[b][:, feat_cols])
        return accuracy_score(y_tr[b], p), f1_score(y_tr[b], p, average="macro"), amp_nonamp_f1(y_tr[b], p)
    return np.array(Parallel(n_jobs=-1)(delayed(run)(a, b) for a, b in SPLITS))

def corrected_ttest(a, b):
    d = a - b; k = len(d); v = d.var(ddof=1)
    if v == 0:
        return np.nan, 1.0
    t = d.mean() / np.sqrt((1 / k + 1 / (CV_FOLDS - 1)) * v)
    return t, 2 * stats.t.sf(abs(t), k - 1)

## 8. Feature ablation (training set, fixed classifier)

The classifier is held fixed (soft-voting RBF-SVM + LightGBM + Extra Trees with standard settings) so that
**only the features change**. 15 paired CV folds; each set is compared with the paper's 45 features.
The proposed feature set is the SISH-aware set (containing stain and/or signal features) with the best
CV accuracy — chosen on the training set only, before the test set is opened.

In [ ]:
def soft_vote(svm=None, lgbm=None, et=None):
    return VotingClassifier([
        ("svm", svm if svm is not None else Pipeline([("scaler", PowerTransformer()),
                 ("clf", SVC(C=10, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_STATE))])),
        ("lgbm", lgbm if lgbm is not None else LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                 subsample=0.85, subsample_freq=1, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=1, verbose=-1)),
        ("et", et if et is not None else ExtraTreesClassifier(n_estimators=600, random_state=RANDOM_STATE, n_jobs=1)),
    ], voting="soft")

def with_selection(k):
    return lambda: Pipeline([("select", SelectKBest(lambda X, y: mutual_info_classif(X, y, random_state=RANDOM_STATE), k=k)),
                             ("model", soft_vote())])

ABLATION = {
    "Paper (45)  [baseline]":          (["paper"], None),
    "Paper + Stain":                   (["paper", "stain"], None),
    "Paper + Fine LBP":                (["paper", "fine_lbp"], None),
    "Paper + SISH signals":            (["paper", "signal"], None),
    "Paper + Stain + Fine LBP":        (["paper", "stain", "fine_lbp"], None),
    "SISH signals only":               (["signal"], None),
    "Stain + SISH signals":            (["stain", "signal"], None),
    "All 86 (SISH fusion)":            (["paper", "stain", "fine_lbp", "signal"], None),
    "All 86 + MI top-40 (in-CV)":      (["paper", "stain", "fine_lbp", "signal"], 40),
}
abl_scores, abl_rows = {}, []
for name, (grp, k) in ABLATION.items():
    t0 = time.time()
    s = repeated_cv(with_selection(k) if k else soft_vote, cols(grp))
    abl_scores[name] = s
    print(f"{name:<34} acc {s[:,0].mean()*100:6.2f} ± {s[:,0].std()*100:4.2f}   macro-F1 {s[:,1].mean()*100:6.2f}"
          f"   Amp-vs-NonAmp F1 {np.nanmean(s[:,2])*100:6.2f}   ({time.time()-t0:.0f}s)")

base = abl_scores["Paper (45)  [baseline]"][:, 0]
for name, (grp, k) in ABLATION.items():
    s = abl_scores[name]; t, p = corrected_ttest(s[:, 0], base)
    abl_rows.append({"Feature set": name, "# features": k or len(cols(grp)),
                     "CV Accuracy (%)": s[:, 0].mean() * 100, "± std": s[:, 0].std() * 100,
                     "Macro F1 (%)": s[:, 1].mean() * 100, "Amp vs Non-Amp F1 (%)": np.nanmean(s[:, 2]) * 100,
                     "Δ vs paper (pp)": (s[:, 0].mean() - base.mean()) * 100, "p-value": p})
ablation = pd.DataFrame(abl_rows).set_index("Feature set").round(4)
display(ablation.style.format(precision=2).format({"p-value": "{:.4f}"}))
ablation.to_csv(os.path.join(WORK_DIR, "sish_ablation.csv"))

# The proposed model must be SISH-aware: choose among sets that contain stain and/or signal features
sish_sets = [n for n, (grp, k) in ABLATION.items() if {"stain", "signal"} & set(grp)]
best_set = ablation.loc[sish_sets, "CV Accuracy (%)"].idxmax()
PROPOSED_GROUPS, PROPOSED_K = ABLATION[best_set]
PROPOSED_COLS = cols(PROPOSED_GROUPS)
print(f"\nProposed feature set (best CV): {best_set}  ->  groups {PROPOSED_GROUPS}, selection k = {PROPOSED_K}")

In [ ]:
o = ablation.sort_values("CV Accuracy (%)")
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#90a4ae" if "baseline" in n else ("#2e7d32" if n == best_set else "#9ec5e8") for n in o.index]
ax.barh(o.index, o["CV Accuracy (%)"], xerr=o["± std"], color=colors, capsize=3)
for i, (v, p) in enumerate(zip(o["CV Accuracy (%)"], o["p-value"])):
    ax.text(v + 0.6, i, f"{v:.1f}%" + ("  *" if p < 0.05 and v > base.mean() * 100 else ""), va="center", fontsize=9)
ax.set_xlim(max(0, o["CV Accuracy (%)"].min() - 15), 100)
ax.set_xlabel("Repeated-CV accuracy (%)   (* = significantly better than paper features, p < 0.05)")
ax.set_title("Feature ablation — same classifier, different features")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); savefig("ablation.png"); plt.show()

## 9. Final models — equal tuning budget

| Model | Features | Role |
|---|---|---|
| Decision Tree, Random Forest, Gradient Boosting, XGBoost | paper 45 | **base paper** |
| Soft-Vote (SVM + LightGBM + ET) | paper 45 | previous best (benchmark notebook) |
| **SISH-Fusion Soft-Vote** | proposed set | **proposed** |
| SISH-Fusion Two-Stage | proposed set | variant: Normal-vs-tumour, then Amplified-vs-Non-amplified specialist |

Every model / ensemble member gets `N_ITER` random configurations scored by group-aware 5-fold CV on the
training set.

In [ ]:
obj = "multi:softprob" if n_classes > 2 else "binary:logistic"
SCALERS = [StandardScaler(), PowerTransformer()]
SPACES = {
    "Decision Tree": (Pipeline([("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
        {"clf__max_depth": [None, 5, 8, 12, 16, 24], "clf__min_samples_leaf": [1, 2, 5, 10, 20], "clf__criterion": ["gini", "entropy"]}),
    "Random Forest": (Pipeline([("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))]),
        {"clf__n_estimators": [200, 400, 600], "clf__max_depth": [None, 10, 20], "clf__max_features": ["sqrt", 0.3, 0.5], "clf__min_samples_leaf": [1, 2, 4]}),
    "Gradient Boosting": (Pipeline([("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [100, 200, 400], "clf__learning_rate": [0.03, 0.05, 0.1], "clf__max_depth": [2, 3, 5], "clf__subsample": [0.7, 0.85, 1.0]}),
    "XGBoost": (Pipeline([("clf", XGBClassifier(objective=obj, random_state=RANDOM_STATE, n_jobs=1, verbosity=0))]),
        {"clf__n_estimators": [200, 400, 600], "clf__max_depth": [3, 4, 6, 8], "clf__learning_rate": [0.03, 0.05, 0.1],
         "clf__subsample": [0.7, 0.85, 1.0], "clf__colsample_bytree": [0.6, 0.8, 1.0], "clf__min_child_weight": [1, 3, 5]}),
    "svm": (Pipeline([("scaler", PowerTransformer()), ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE))]),
        {"scaler": SCALERS, "clf__C": loguniform(1e-1, 1e3), "clf__gamma": loguniform(1e-4, 1e0)}),
    "lgbm": (Pipeline([("clf", LGBMClassifier(random_state=RANDOM_STATE, n_jobs=1, verbose=-1))]),
        {"clf__n_estimators": [200, 400, 800], "clf__num_leaves": [15, 31, 63], "clf__learning_rate": [0.02, 0.05, 0.1],
         "clf__subsample": [0.7, 0.85, 1.0], "clf__subsample_freq": [1], "clf__colsample_bytree": [0.6, 0.8, 1.0],
         "clf__min_child_samples": [5, 10, 20, 40], "clf__reg_lambda": [0, 1, 5]}),
    "et": (Pipeline([("clf", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1))]),
        {"clf__n_estimators": [300, 600, 900], "clf__max_features": ["sqrt", 0.3, 0.5, None], "clf__min_samples_leaf": [1, 2, 4], "clf__max_depth": [None, 20, 30]}),
}

def tune(key, feat_cols):
    pipe, space = SPACES[key]
    rs = RandomizedSearchCV(pipe, space, n_iter=N_ITER, cv=make_cv(RANDOM_STATE), scoring="accuracy",
                            n_jobs=-1, random_state=RANDOM_STATE, refit=False)
    rs.fit(X_tr[:, feat_cols], y_tr, groups=g_tr)
    return clone(pipe).set_params(**rs.best_params_), rs.best_score_


# ColumnModel (imported from her2_models.py) applies a model to its own columns of the 86-feature matrix


class TwoStage(BaseEstimator, ClassifierMixin):
    # Stage 1: Normal vs tumour.  Stage 2: Amplified vs Non-amplified specialist (trained on tumour only)
    def __init__(self, base=None, normal=None):
        self.base, self.normal = base, normal
    def fit(self, X, y):
        self.classes_ = np.unique(y); t = y != self.normal
        self.s1_ = clone(self.base).fit(X, (y == self.normal).astype(int))
        self.s2_ = clone(self.base).fit(X[t], y[t]); return self
    def predict_proba(self, X):
        pn = self.s1_.predict_proba(X)[:, 1]; p2 = self.s2_.predict_proba(X)
        P = np.zeros((len(X), len(self.classes_))); P[:, self.normal] = pn
        for j, c in enumerate(self.s2_.classes_):
            P[:, c] = (1 - pn) * p2[:, j]
        return P
    def predict(self, X): return self.predict_proba(X).argmax(1)


PAPER_COLS = cols(["paper"])
final, kind, feat_label, search_cv = {}, {}, {}, {}
for name in ["Decision Tree", "Random Forest", "Gradient Boosting", "XGBoost"]:
    t0 = time.time(); m, s = tune(name, PAPER_COLS)
    final[name], kind[name], feat_label[name], search_cv[name] = ColumnModel(m, PAPER_COLS), "Base paper", "paper 45", s
    print(f"[Base paper] {name:<28} search CV {s*100:6.2f}%  ({time.time()-t0:.0f}s)")

members = {}
for fs_name, fcols in [("paper", PAPER_COLS), ("proposed", PROPOSED_COLS)]:
    for key in ["svm", "lgbm", "et"]:
        t0 = time.time(); members[(fs_name, key)], s = tune(key, fcols)
        print(f"[member/{fs_name:<8}] {key:<5} search CV {s*100:6.2f}%  ({time.time()-t0:.0f}s)")

final["Soft-Vote (paper features)"] = ColumnModel(soft_vote(*[members[("paper", k)] for k in ["svm", "lgbm", "et"]]), PAPER_COLS)
kind["Soft-Vote (paper features)"], feat_label["Soft-Vote (paper features)"] = "Previous best", "paper 45"

PROPOSED = "SISH-Fusion Soft-Vote"
prop_vote = soft_vote(*[members[("proposed", k)] for k in ["svm", "lgbm", "et"]])
final[PROPOSED] = ColumnModel(prop_vote, PROPOSED_COLS, PROPOSED_K)
kind[PROPOSED], feat_label[PROPOSED] = "Proposed", best_set

if NORMAL is not None and n_classes == 3:
    final["SISH-Fusion Two-Stage"] = ColumnModel(TwoStage(prop_vote, NORMAL), PROPOSED_COLS, PROPOSED_K)
    kind["SISH-Fusion Two-Stage"], feat_label["SISH-Fusion Two-Stage"] = "Proposed variant", best_set

In [ ]:
# Repeated CV (15 paired folds) for every final model, then fit on the full training set with timing
def repeated_cv_model(model):
    def run(a, b):
        m = clone(model).fit(X_tr[a], y_tr[a]); p = predict_labels(m, X_tr[b])
        return accuracy_score(y_tr[b], p), f1_score(y_tr[b], p, average="macro"), amp_nonamp_f1(y_tr[b], p)
    return np.array(Parallel(n_jobs=-1)(delayed(run)(a, b) for a, b in SPLITS))

cv_final, fitted, train_time = {}, {}, {}
for name, model in final.items():
    cv_final[name] = repeated_cv_model(model)
    t0 = time.perf_counter(); fitted[name] = clone(model).fit(X_tr, y_tr); train_time[name] = time.perf_counter() - t0
    print(f"{name:<30} repeated CV {cv_final[name][:,0].mean()*100:6.2f} ± {cv_final[name][:,0].std()*100:4.2f}%"
          f"   train {train_time[name]:6.1f}s")

BEST_BASELINE = max([n for n in final if kind[n] in ("Base paper", "Previous best")], key=lambda n: cv_final[n][:, 0].mean())
print("\nStrongest baseline (by CV):", BEST_BASELINE)

## 10. Held-out test set: baseline vs proposed
Accuracy, macro precision / recall / F1, ROC-AUC (one-vs-rest), Cohen's κ, bootstrap 95 % CI,
training time and inference time per image.

In [ ]:
def bootstrap_ci(y_true, y_pred, n=2000):
    rng = np.random.default_rng(RANDOM_STATE)
    accs = [accuracy_score(y_true[s], y_pred[s]) for s in (rng.integers(0, len(y_true), len(y_true)) for _ in range(n))]
    return np.percentile(accs, [2.5, 97.5]) * 100

pred, proba, rows = {}, {}, []
for name, m in fitted.items():
    t0 = time.perf_counter(); pr = m.predict_proba(X_te); infer_ms = (time.perf_counter() - t0) / len(X_te) * 1000
    p = pr.argmax(1); pred[name], proba[name] = p, pr
    lo, hi = bootstrap_ci(y_te, p)
    auc = roc_auc_score(y_te, pr, multi_class="ovr", average="macro") if n_classes > 2 else roc_auc_score(y_te, pr[:, 1])
    rows.append({"Model": name, "Role": kind[name], "Features": feat_label[name],
                 "Accuracy (%)": accuracy_score(y_te, p) * 100, "95% CI": f"{lo:.1f}–{hi:.1f}",
                 "Precision (%)": precision_score(y_te, p, average="macro", zero_division=0) * 100,
                 "Recall (%)": recall_score(y_te, p, average="macro", zero_division=0) * 100,
                 "F1 (%)": f1_score(y_te, p, average="macro") * 100,
                 "Amp vs Non-Amp F1 (%)": amp_nonamp_f1(y_te, p) * 100,
                 "ROC-AUC": auc, "Kappa": cohen_kappa_score(y_te, p),
                 "CV Acc (%)": cv_final[name][:, 0].mean() * 100,
                 "Train time (s)": train_time[name], "Inference (ms/img)": infer_ms})
results = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy (%)", ascending=False)
results.to_csv(os.path.join(WORK_DIR, "sish_final_comparison.csv"))

def hl(r):
    if r.name == PROPOSED: return ["background-color:#c8e6c9;font-weight:bold"] * len(r)
    if r.name == BEST_BASELINE: return ["background-color:#eceff1"] * len(r)
    return [""] * len(r)
display(results.style.apply(hl, axis=1).format(precision=2))

# Significance: proposed vs every baseline
sig = []
for b in [n for n in final if kind[n] in ("Base paper", "Previous best")]:
    t, p_cv = corrected_ttest(cv_final[PROPOSED][:, 0], cv_final[b][:, 0])
    a_ok, b_ok = pred[PROPOSED] == y_te, pred[b] == y_te
    n10, n01 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    p_mc = stats.binomtest(min(n10, n01), n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    sig.append({"Proposed vs": b, "Δ test acc (pp)": (a_ok.mean() - b_ok.mean()) * 100,
                "Δ CV acc (pp)": (cv_final[PROPOSED][:, 0].mean() - cv_final[b][:, 0].mean()) * 100,
                "CV t-test p": p_cv, "McNemar: proposed right / baseline right": f"{n10} / {n01}", "McNemar p": p_mc})
sig = pd.DataFrame(sig).set_index("Proposed vs")
display(sig.style.format(precision=4))
sig.to_csv(os.path.join(WORK_DIR, "sish_significance.csv"))

In [ ]:
# Charts: accuracy with CI, metric groups, confusion matrices, ROC curves
o = results.sort_values("Accuracy (%)")
ci = np.array([[float(v) for v in s.split("–")] for s in o["95% CI"]])
col = ["#2e7d32" if kind[n] == "Proposed" else ("#9ec5e8" if kind[n] == "Proposed variant" else "#90a4ae") for n in o.index]
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(o.index, o["Accuracy (%)"], color=col,
        xerr=np.vstack([o["Accuracy (%)"] - ci[:, 0], ci[:, 1] - o["Accuracy (%)"]]), capsize=3)
for i, v in enumerate(o["Accuracy (%)"]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(max(0, ci[:, 0].min() - 5), 101); ax.set_xlabel("Test accuracy (%) with bootstrap 95% CI")
ax.set_title("Baselines (grey) vs proposed SISH-fusion (green)"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); savefig("test_accuracy.png"); plt.show()

met = ["Accuracy (%)", "Precision (%)", "Recall (%)", "F1 (%)", "Amp vs Non-Amp F1 (%)"]
cmp = results.loc[[BEST_BASELINE, PROPOSED], met].T
ax = cmp.plot(kind="bar", figsize=(10, 4.2), rot=0, color=["#90a4ae", "#2e7d32"], width=0.75)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.1f", fontsize=8)
ax.set_ylim(max(0, cmp.values.min() - 15), 100); ax.set_title(f"{BEST_BASELINE} vs {PROPOSED}")
ax.legend(loc="lower right"); plt.tight_layout(); savefig("baseline_vs_proposed_metrics.png"); plt.show()

names = list(results.index)
nc = 3; nr = int(np.ceil(len(names) / nc))
fig, axes = plt.subplots(nr, nc, figsize=(4.8 * nc, 4.3 * nr)); axes = np.array(axes).ravel()
for ax, n in zip(axes, names):
    ConfusionMatrixDisplay(confusion_matrix(y_te, pred[n]), display_labels=[c[:12] for c in CLASS_NAMES]).plot(
        ax=ax, colorbar=False, cmap="Greens" if kind[n].startswith("Proposed") else "Greys")
    ax.set_title(f"{n}\n{results.loc[n, 'Accuracy (%)']:.1f}%", fontsize=10); ax.tick_params(axis="x", rotation=30)
for ax in axes[len(names):]: ax.axis("off")
plt.tight_layout(); savefig("confusion_matrices.png"); plt.show()

Yb = label_binarize(y_te, classes=range(n_classes))
if n_classes == 2:
    Yb = np.c_[1 - Yb, Yb]
fig, axes = plt.subplots(1, n_classes, figsize=(4.6 * n_classes, 4.2))
for k, ax in enumerate(np.atleast_1d(axes)):
    for n, c, ls in [(BEST_BASELINE, "#78909c", "--"), (PROPOSED, "#2e7d32", "-")]:
        fpr, tpr, _ = roc_curve(Yb[:, k], proba[n][:, k])
        ax.plot(fpr, tpr, ls, color=c, label=f"{n[:26]} (AUC {roc_auc_score(Yb[:, k], proba[n][:, k]):.3f})")
    ax.plot([0, 1], [0, 1], ":", c="grey"); ax.set_title(f"ROC — {CLASS_NAMES[k]}"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); savefig("roc_curves.png"); plt.show()

print(f"Classification report — {PROPOSED}")
print(classification_report(y_te, pred[PROPOSED], target_names=CLASS_NAMES, digits=4))
print(f"Classification report — {BEST_BASELINE}")
print(classification_report(y_te, pred[BEST_BASELINE], target_names=CLASS_NAMES, digits=4))

In [ ]:
# Per-class precision / recall / F1 for every model (report table) + baseline-vs-proposed chart
pc_rows = []
for n in results.index:
    rep_ = classification_report(y_te, pred[n], target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    for c in CLASS_NAMES:
        pc_rows.append({"Model": n, "Class": c, "Precision (%)": rep_[c]["precision"] * 100,
                        "Recall (%)": rep_[c]["recall"] * 100, "F1 (%)": rep_[c]["f1-score"] * 100, "Support": rep_[c]["support"]})
per_class = pd.DataFrame(pc_rows).set_index(["Model", "Class"]).round(2)
per_class.to_csv(os.path.join(WORK_DIR, "sish_per_class.csv"))
display(per_class.loc[[PROPOSED, BEST_BASELINE]])

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, met_ in zip(axes, ["Precision (%)", "Recall (%)", "F1 (%)"]):
    d = per_class[met_].unstack(0)[[BEST_BASELINE, PROPOSED]]
    d.index = [c[:16] for c in d.index]
    d.plot(kind="bar", ax=ax, rot=0, color=["#90a4ae", "#2e7d32"], width=0.75, legend=(met_ == "F1 (%)"))
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.0f", fontsize=8)
    ax.set_title(met_); ax.set_ylim(0, 108); ax.spines[["top", "right"]].set_visible(False)
axes[-1].legend([BEST_BASELINE[:28], PROPOSED[:28]], fontsize=8, loc="lower right")
plt.suptitle("Per-class performance on the held-out test set"); plt.tight_layout(); savefig("per_class_metrics.png"); plt.show()

### 10.1 Feature-extraction cost
The signal features run at native resolution, so they cost more time than the paper's 128 × 128 features.

In [ ]:
def paper_only(path):
    img = Image.open(path).convert("RGB").resize(F.IMAGE_SIZE)
    rgb, gray = np.asarray(img), np.asarray(img.convert("L"))
    return np.concatenate([F.extract_color_features(rgb), F.extract_lbp_features(gray), F.extract_haralick_features(gray)])

sample_paths = list(pd.Series(paths).sample(min(20, len(paths)), random_state=0))
t0 = time.perf_counter(); [paper_only(p) for p in sample_paths]; t_paper = (time.perf_counter() - t0) / len(sample_paths)
t0 = time.perf_counter(); [F.extract_feature_vector(p) for p in sample_paths]; t_sish = (time.perf_counter() - t0) / len(sample_paths)
timing = pd.DataFrame({
    "Feature extraction (ms/img)": [t_paper * 1000, t_sish * 1000],
    "Model inference (ms/img)": [results.loc[BEST_BASELINE, "Inference (ms/img)"], results.loc[PROPOSED, "Inference (ms/img)"]],
    "Training time (s)": [results.loc[BEST_BASELINE, "Train time (s)"], results.loc[PROPOSED, "Train time (s)"]],
}, index=[f"Baseline: {BEST_BASELINE}", f"Proposed: {PROPOSED}"]).round(2)
timing["Total per image (ms)"] = timing["Feature extraction (ms/img)"] + timing["Model inference (ms/img)"]
display(timing); timing.to_csv(os.path.join(WORK_DIR, "sish_timing.csv"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
timing[["Feature extraction (ms/img)", "Model inference (ms/img)"]].plot(kind="barh", stacked=True, ax=axes[0], color=["#9ec5e8", "#2e7d32"])
axes[0].set_xlabel("ms per image"); axes[0].set_title("Prediction time per image"); axes[0].legend(fontsize=8)
timing["Training time (s)"].plot(kind="barh", ax=axes[1], color="#90a4ae"); axes[1].set_xlabel("seconds"); axes[1].set_title("Training time (full training set)")
for ax in axes: ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); savefig("timing.png"); plt.show()

## 11. Reliability: conformal prediction (base paper's framework)
Cross-conformal calibration (out-of-fold probabilities on the training set) at α = 0.05 and 0.10. The same final
model gives both the prediction and its prediction set. Lower *Uncertain %* with coverage ≥ 1 − α is better.

In [ ]:
def conformal(model_name):
    oof = np.zeros((len(y_tr), n_classes))
    for a, b in make_cv(7).split(X_tr, y_tr, g_tr):
        oof[b] = clone(final[model_name]).fit(X_tr[a], y_tr[a]).predict_proba(X_tr[b])
    s = 1 - oof[np.arange(len(y_tr)), y_tr]; n = len(s)
    out = []
    for a in ALPHAS:
        q = np.quantile(s, min(1, np.ceil((n + 1) * (1 - a)) / n), method="higher")
        sets = proba[model_name] >= 1 - q; size = sets.sum(1); single = size == 1
        out.append({"Model": model_name, "alpha": a, "Coverage (%)": sets[np.arange(len(y_te)), y_te].mean() * 100,
                    "Avg set size": size.mean(), "Uncertain → ISH (%)": (~single).mean() * 100,
                    "Accuracy when confident (%)": (sets[single].argmax(1) == y_te[single]).mean() * 100 if single.any() else np.nan})
    return out

cp = pd.DataFrame([r for n in final for r in conformal(n)]).set_index(["alpha", "Model"]).round(2)
for a in ALPHAS:
    print(f"\n=== alpha = {a} (target coverage {(1-a)*100:.0f}%) ===")
    display(cp.loc[a].style.apply(lambda r: ["background-color:#c8e6c9;font-weight:bold" if r.name == PROPOSED else ""] * len(r), axis=1).format(precision=2))
cp.to_csv(os.path.join(WORK_DIR, "sish_conformal.csv"))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, met_ in zip(axes, ["Coverage (%)", "Uncertain → ISH (%)", "Accuracy when confident (%)"]):
    d = cp[met_].unstack(1)[[BEST_BASELINE, PROPOSED]]
    d.index = [f"α = {a}" for a in d.index]
    d.plot(kind="bar", ax=ax, rot=0, color=["#90a4ae", "#2e7d32"], width=0.7, legend=False)
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.1f", fontsize=8)
    if met_ == "Coverage (%)":
        for i, a in enumerate(ALPHAS):
            ax.hlines((1 - a) * 100, i - 0.4, i + 0.4, colors="r", linestyles="--", lw=1)
    ax.set_title(met_); ax.set_ylim(0, 108); ax.spines[["top", "right"]].set_visible(False)
axes[2].legend([BEST_BASELINE[:28], PROPOSED[:28]], fontsize=8, loc="lower right")
plt.suptitle("Conformal prediction: baseline vs proposed (red dashes = target coverage)"); plt.tight_layout()
savefig("conformal_comparison.png"); plt.show()

## 12. Which feature groups does the proposed model rely on?
Grouped permutation importance on the test set: all columns of a group are shuffled together, and the drop in
accuracy is measured (10 repeats).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
base_acc = accuracy_score(y_te, pred[PROPOSED])
imp = []
for gname in ["paper", "stain", "fine_lbp", "signal"]:
    gc = cols([gname])
    if not np.isin(gc, PROPOSED_COLS).any():
        continue
    drops = []
    for _ in range(10):
        Xp = X_te.copy(); perm = rng.permutation(len(Xp)); Xp[:, gc] = Xp[perm][:, gc]
        drops.append(base_acc - accuracy_score(y_te, predict_labels(fitted[PROPOSED], Xp)))
    imp.append({"Feature group": gname, "Accuracy drop (pp)": np.mean(drops) * 100, "± std": np.std(drops) * 100})
imp = pd.DataFrame(imp).set_index("Feature group").sort_values("Accuracy drop (pp)")
display(imp.round(2))
imp["Accuracy drop (pp)"].plot(kind="barh", xerr=imp["± std"], color="#2e7d32", figsize=(7, 3), capsize=3)
plt.xlabel("Accuracy drop when the group is shuffled (pp)"); plt.title(f"Grouped permutation importance — {PROPOSED}")
plt.tight_layout(); savefig("group_importance.png"); plt.show()
imp.to_csv(os.path.join(WORK_DIR, "sish_group_importance.csv"))

## 13. Summary of the contribution (numbers filled in from this run)

In [ ]:
rb, rp = results.loc[BEST_BASELINE], results.loc[PROPOSED]
cpb, cpp = cp.loc[(0.10, BEST_BASELINE)], cp.loc[(0.10, PROPOSED)]
s = sig.loc[BEST_BASELINE]
lines = [
    ("CV accuracy (%)", rb["CV Acc (%)"], rp["CV Acc (%)"]),
    ("Test accuracy (%)", rb["Accuracy (%)"], rp["Accuracy (%)"]),
    ("Macro precision (%)", rb["Precision (%)"], rp["Precision (%)"]),
    ("Macro recall (%)", rb["Recall (%)"], rp["Recall (%)"]),
    ("Macro F1 (%)", rb["F1 (%)"], rp["F1 (%)"]),
    ("Amp vs Non-Amp F1 (%)", rb["Amp vs Non-Amp F1 (%)"], rp["Amp vs Non-Amp F1 (%)"]),
    ("ROC-AUC", rb["ROC-AUC"], rp["ROC-AUC"]),
    ("CP@0.10 flagged for ISH (%)", cpb["Uncertain → ISH (%)"], cpp["Uncertain → ISH (%)"]),
    ("CP@0.10 acc. when confident (%)", cpb["Accuracy when confident (%)"], cpp["Accuracy when confident (%)"]),
]
print("=" * 84)
print(f" Baseline : {BEST_BASELINE} (paper features)")
print(f" Proposed : {PROPOSED} ({best_set})")
print("-" * 84)
print(f" {'':34}{'Baseline':>14}{'Proposed':>14}{'Change':>14}")
for lbl, b_, p_ in lines:
    print(f" {lbl:<34}{b_:>14.3f}{p_:>14.3f}{p_-b_:>+14.3f}")
print("-" * 84)
print(f" CV corrected t-test p = {s['CV t-test p']:.4f}   |   McNemar p = {s['McNemar p']:.4f}")
verdict = ("significant" if (s["CV t-test p"] < 0.05 and s["Δ CV acc (pp)"] > 0) else "NOT significant")
print(f" -> The CV improvement is {verdict} at the 5% level.")
print("=" * 84)

## 14. Export the trained models for the Flask website

Builds `her2_sish_models.joblib` — **the exact file `her2_flask/app.py` loads** — from the models trained above,
on the same train/test split, with their test metrics and conformal thresholds. XGBoost and the two-stage
variant are left out so the website needs only scikit-learn and LightGBM.

Model files only load reliably with the **same library versions** they were saved with, so this cell also writes
`requirements_web.txt` with the exact versions used here.

In [ ]:
import sklearn, lightgbm, scipy, platform, datetime as dt

def cross_conformal_q(model):
    oof = np.zeros((len(y_tr), n_classes))
    for a, b in make_cv(7).split(X_tr, y_tr, g_tr):
        oof[b] = clone(model).fit(X_tr[a], y_tr[a]).predict_proba(X_tr[b])
    s_ = 1 - oof[np.arange(len(y_tr)), y_tr]; n_ = len(s_)
    return {a: float(np.quantile(s_, min(1, np.ceil((n_ + 1) * (1 - a)) / n_), method="higher")) for a in ALPHAS}

def web_row(name, model, cv_scores, role, train_s):
    t0 = time.perf_counter(); pr = model.predict_proba(X_te); infer_ms = (time.perf_counter() - t0) / len(X_te) * 1000
    p = pr.argmax(1); q = cross_conformal_q(model)
    sets = pr >= 1 - q[0.10]; single = sets.sum(1) == 1
    lo, hi = bootstrap_ci(y_te, p)
    auc = roc_auc_score(y_te, pr, multi_class="ovr", average="macro") if n_classes > 2 else roc_auc_score(y_te, pr[:, 1])
    a_f1 = amp_nonamp_f1(y_te, p)
    row = {"model": name, "role": role, "main": name == PROPOSED,
           "n_features": int(model.k_select or len(model.feat_cols)),
           "test_acc": round(accuracy_score(y_te, p) * 100, 2), "ci": [round(lo, 1), round(hi, 1)],
           "precision": round(precision_score(y_te, p, average="macro", zero_division=0) * 100, 2),
           "recall": round(recall_score(y_te, p, average="macro", zero_division=0) * 100, 2),
           "macro_f1": round(f1_score(y_te, p, average="macro") * 100, 2),
           "amp_f1": None if np.isnan(a_f1) else round(a_f1 * 100, 2),
           "auc": round(float(auc), 3), "cv_acc": round(cv_scores[:, 0].mean() * 100, 2),
           "cp_coverage": round(sets[np.arange(len(y_te)), y_te].mean() * 100, 1),
           "cp_uncertain": round((~single).mean() * 100, 1),
           "cp_confident_acc": round((sets[single].argmax(1) == y_te[single]).mean() * 100, 1) if single.any() else None,
           "train_s": round(train_s, 2), "infer_ms": round(infer_ms, 3)}
    return row, q

WEB_BASELINE = "Soft-Vote (paper features)"
web_models, web_rows, web_q = {}, [], {}

def add(name, model, cv_scores, role, train_s):
    row, q = web_row(name, model, cv_scores, role, train_s)
    web_models[name], web_q[name] = model, q
    web_rows.append(row)
    print(f"  {name:<38} test {row['test_acc']:6.2f}%   CV {row['cv_acc']:6.2f}%   flagged for ISH {row['cp_uncertain']:5.1f}%")

add(PROPOSED, fitted[PROPOSED], cv_final[PROPOSED], "Proposed", train_time[PROPOSED])
for key, label in [("svm", "RBF-SVM"), ("lgbm", "LightGBM"), ("et", "Extra Trees")]:
    spec = ColumnModel(members[("proposed", key)], PROPOSED_COLS, PROPOSED_K)
    t0 = time.perf_counter(); m = clone(spec).fit(X_tr, y_tr); ts = time.perf_counter() - t0
    add(f"{label} (SISH features)", m, repeated_cv_model(spec), "Proposed member", ts)
add(WEB_BASELINE, fitted[WEB_BASELINE], cv_final[WEB_BASELINE], "Baseline", train_time[WEB_BASELINE])
for n in ["Random Forest", "Gradient Boosting", "Decision Tree"]:
    add(f"{n} (paper features)", fitted[n], cv_final[n], "Base paper", train_time[n])

versions = {"python": platform.python_version(), "numpy": np.__version__, "scikit-learn": sklearn.__version__,
            "lightgbm": lightgbm.__version__, "scipy": scipy.__version__}
bundle = {
    "models": web_models, "main": PROPOSED, "baseline": WEB_BASELINE, "qhat": web_q, "classes": CLASS_NAMES,
    "metrics": sorted(web_rows, key=lambda r: (not r["main"], r["role"] != "Baseline", -r["test_acc"])),
    "proposed_groups": list(PROPOSED_GROUPS), "proposed_k": int(PROPOSED_K or 0),
    "report": classification_report(y_te, pred[PROPOSED], target_names=CLASS_NAMES, digits=4),
    "confusion_matrix": confusion_matrix(y_te, pred[PROPOSED]).tolist(),
    "n_train": int(len(tr_idx)), "n_test": int(len(te_idx)),
    "trained_at": dt.datetime.now().strftime("%Y-%m-%d %H:%M") + " (Kaggle)", "versions": versions,
    "sklearn_version": sklearn.__version__,
}
WEB_DIR = os.path.join(WORK_DIR, "flask_models"); os.makedirs(WEB_DIR, exist_ok=True)
joblib.dump(bundle, os.path.join(WEB_DIR, "her2_sish_models.joblib"), compress=3)
np.savez(os.path.join(WEB_DIR, "her2_sish_features.npz"),
         X=X_all, y=y_raw, g=groups, paths=np.array([os.path.basename(p) for p in paths]),
         feature_names=np.array(F.FEATURE_NAMES), proposed_groups=np.array(PROPOSED_GROUPS),
         proposed_k=np.array(PROPOSED_K if PROPOSED_K else 0), test_idx=te_idx)
with open(os.path.join(WEB_DIR, "requirements_web.txt"), "w") as f:
    f.write("# Exact versions the models were trained with (Python " + versions["python"] + ")\n"
            f"flask\nnumpy=={versions['numpy']}\npandas\nscipy\npillow\nscikit-image\nmahotas\njoblib\n"
            f"scikit-learn=={versions['scikit-learn']}\nlightgbm=={versions['lightgbm']}\n")

# Sanity check: reload the bundle the way the website does and predict one test image
re_b = joblib.load(os.path.join(WEB_DIR, "her2_sish_models.joblib"))
chk = re_b["models"][PROPOSED].predict_proba(X_te[:1])[0]
print("\nReloaded OK:", CLASS_NAMES[int(chk.argmax())], f"({chk.max()*100:.1f}%)  | versions:", versions)
print("Saved", os.path.join(WEB_DIR, "her2_sish_models.joblib"),
      f"({os.path.getsize(os.path.join(WEB_DIR, 'her2_sish_models.joblib'))/1e6:.1f} MB)")

## 15. Report pack: every table in one Excel file + one zip to download

`sish_results.zip` contains

| Folder | Contents |
|---|---|
| `figures/` | all charts at 300 dpi (ablation, accuracy, metrics, per-class, confusion matrices, ROC, conformal, timing, signal features, detection examples, importance) |
| `tables/` | every CSV + `report_tables.xlsx` (one sheet per table) |
| `flask_models/` | `her2_sish_models.joblib`, `her2_sish_features.npz`, `requirements_web.txt` → copy into `her2_flask/` |
| `code/` | `sish_features.py`, `her2_models.py` (identical to the website's copies) |

In [ ]:
TABLES = {
    "Ablation": "sish_ablation.csv", "Final comparison": "sish_final_comparison.csv",
    "Per-class": "sish_per_class.csv", "Significance": "sish_significance.csv",
    "Conformal": "sish_conformal.csv", "Timing": "sish_timing.csv",
    "Group importance": "sish_group_importance.csv", "Signal feature tests": "sish_signal_feature_tests.csv",
}
xlsx = os.path.join(WORK_DIR, "report_tables.xlsx")
try:
    with pd.ExcelWriter(xlsx) as xw:
        for sheet, f in TABLES.items():
            fp = os.path.join(WORK_DIR, f)
            if os.path.exists(fp):
                pd.read_csv(fp).to_excel(xw, sheet_name=sheet[:31], index=False)
    print("Saved", xlsx)
except Exception as e:
    print("Excel export skipped:", e)

zpath = os.path.join(WORK_DIR, "sish_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in list(TABLES.values()) + ["report_tables.xlsx"]:
        if os.path.exists(os.path.join(WORK_DIR, f)):
            z.write(os.path.join(WORK_DIR, f), "tables/" + f)
    for f in sorted(os.listdir(FIG_DIR)):
        z.write(os.path.join(FIG_DIR, f), "figures/" + f)
    for f in sorted(os.listdir(WEB_DIR)):
        z.write(os.path.join(WEB_DIR, f), "flask_models/" + f)
    for f in ["sish_features.py", "her2_models.py"]:
        z.write(f, "code/" + f)
print(f"Saved {zpath} ({os.path.getsize(zpath)/1e6:.1f} MB)")
for info in zipfile.ZipFile(zpath).infolist():
    print(f"  {info.filename:<48} {info.file_size/1e3:8.1f} KB")
if IN_COLAB:
    from google.colab import files as _f; _f.download(zpath)
else:
    print("\nKaggle: open the Output panel (right side) -> /kaggle/working -> download sish_results.zip")

## 16. Predict a single image (with signal overlay)

In [ ]:
from PIL import Image as PILImage

if IN_COLAB:
    from google.colab import files
    print("Upload one HER2 image")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Please upload exactly ONE image.")
    image_name = list(uploaded.keys())[0]
    image_path = os.path.join("/content", image_name)
    print("Uploaded image:", image_name)
    display(PILImage.open(image_path))
else:
    import ipywidgets as widgets
    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    print("Click 'Upload', choose ONE HER2 image, then run the next cell.")
    display(uploader)

In [ ]:
if not IN_COLAB:
    if not uploader.value:
        raise ValueError("Please upload an image first!")

    # Support different ipywidgets versions
    uploaded = uploader.value
    if isinstance(uploaded, dict):          # ipywidgets 7.x
        file_info = list(uploaded.values())[0]
        image_name = file_info["metadata"]["name"] if "metadata" in file_info else file_info["name"]
    else:                                   # ipywidgets 8.x
        file_info = uploaded[0]
        image_name = file_info["name"]
    image_bytes = bytes(file_info["content"])

    image_path = os.path.join(WORK_DIR, image_name)
    with open(image_path, "wb") as f:
        f.write(image_bytes)

    print("Uploaded image:", image_name)
    print("Saved at:", image_path)
    display(PILImage.open(image_path))

In [ ]:
v = F.extract_feature_vector(image_path).reshape(1, -1)
ov, m = F.signal_overlay(image_path)
out = []
for n in [PROPOSED, BEST_BASELINE]:
    p = fitted[n].predict_proba(v)[0]
    out.append({"Model": n, "Prediction": CLASS_NAMES[int(p.argmax())], "Probability (%)": round(p.max() * 100, 2),
                **{f"P({c}) %": round(q * 100, 2) for c, q in zip(CLASS_NAMES, p)}})
display(pd.DataFrame(out).set_index("Model"))
print(f"HER2 signals: {m['her2_signals']:.0f} | CEP17 signals: {m['cep17_signals']:.0f} | "
      f"HER2/CEP17 ratio: {m['her2_cep17_ratio']:.2f} (automated estimate; clinical cut-off is 2.0)")
plt.figure(figsize=(9, 6)); plt.imshow(ov); plt.axis("off")
plt.title("Green = HER2 signals, yellow = CEP17 signals"); plt.show()